# Data Preparation Lab -- Module 2, Class 2

**Dataset:** Superstore Sales

In this lab you will:
1. Load and inspect data
2. Handle missing values
3. Remove duplicates
4. Convert data types
5. Create derived features

The first 3 tasks are pre-built. The rest are TODO for you.

---
## Setup: Load the Dataset

Option A: Upload from Kaggle (download from https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

Option B: Use the URL loader below.

In [1]:
import pandas as pd
import numpy as np

# Option A: Upload in Colab
# from google.colab import files
# uploaded = files.upload()  # upload your CSV
# df = pd.read_csv('SampleSuperstore.csv')

# Option B: Load from a public URL
# If the URL does not work, use Option A.
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/superstore.csv"

try:
    df = pd.read_csv(url, encoding='latin-1')
    print(f"Loaded from URL: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception as e:
    print(f"URL failed ({e}). Use Option A: upload the CSV manually.")
    # Fallback: upload manually
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename, encoding='latin-1')
    print(f"Loaded from upload: {df.shape[0]} rows, {df.shape[1]} columns")

URL failed (HTTP Error 404: Not Found). Use Option A: upload the CSV manually.


Saving SampleSuperstore.csv to SampleSuperstore.csv
Loaded from upload: 10800 rows, 21 columns


---
## Task 1: Inspect the Data (pre-built)

Always look at your data before doing anything to it.

In [2]:
# First 5 rows
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164


In [3]:
# Shape: rows x columns
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Shape: 10800 rows, 21 columns


In [4]:
# Data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10800 entries, 0 to 10799
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         10800 non-null  object 
 1   Order ID       10800 non-null  object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9983 non-null   float64
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quanti

In [5]:
# Summary statistics for numerical columns
df.describe()

,Postal Code,Sales,Quantity,Discount,Profit
count,9983.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,55245.233297,229.858001,3.789574,0.156203,28.656896
std,32038.715955,623.245101,2.225110,0.206452,234.260108
min,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,57103.000000,54.490000,3.000000,0.200000,8.666500
75%,90008.000000,209.940000,5.000000,0.200000,29.364000
max,99301.000000,22638.480000,14.000000,0.800000,8399.976000


---
## Task 2: Missing Values (pre-built)

Check which columns have missing values and how many.

In [6]:
# Count missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})

# Show only columns with missing values
missing_report[missing_report['missing_count'] > 0]

,missing_count,missing_pct
Order Date,806,7.46
Ship Date,806,7.46
Ship Mode,806,7.46
Customer ID,806,7.46
Customer Name,806,7.46
Segment,806,7.46
Country,806,7.46
City,806,7.46
State,806,7.46
Postal Code,817,7.56


In [7]:
# Fill missing numerical values with median (robust to outliers)
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled {col} missing values with median: {median_val}")

# Fill missing categorical values with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled {col} missing values with mode: {mode_val}")

# Verify: no more missing values
print(f"\nTotal missing values remaining: {df.isnull().sum().sum()}")

Filled Postal Code missing values with median: 57103.0
Filled Sales missing values with median: 54.489999999999995
Filled Quantity missing values with median: 3.0
Filled Discount missing values with median: 0.2
Filled Profit missing values with median: 8.6665
Filled Order Date missing values with mode: 9/5/2017
Filled Ship Date missing values with mode: 12/16/2016
Filled Ship Mode missing values with mode: Standard Class
Filled Customer ID missing values with mode: WB-21850
Filled Customer Name missing values with mode: William Brown
Filled Segment missing values with mode: Consumer
Filled Country missing values with mode: United States
Filled City missing values with mode: New York City
Filled State missing values with mode: California
Filled Region missing values with mode: West
Filled Product ID missing values with mode: OFF-PA-10001970
Filled Category missing values with mode: Office Supplies
Filled Sub-Category missing values with mode: Binders
Filled Product Name missing values w

/tmp/ipykernel_2149/1003425343.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)
/tmp/ipykernel_2149/1003425343.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using '

---
## Task 3: Remove Duplicates (pre-built)

In [8]:
# Check for duplicates
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {n_duplicates}")

# Remove duplicates
if n_duplicates > 0:
    df = df.drop_duplicates()
    print(f"After removal: {df.shape[0]} rows remain")
else:
    print("No duplicates to remove.")

Duplicate rows found: 504
After removal: 10296 rows remain


---
## Task 4: Convert Date Columns (TODO)

The `Order Date` and `Ship Date` columns are stored as strings. Convert them to proper datetime objects.

Hint: Use `pd.to_datetime()`. After conversion, verify with `.dtypes`.

In [9]:
# Check current types of date columns
print("Before conversion:")
for col in df.columns:
    if 'date' in col.lower() or 'Date' in col:
        print(f"  {col}: {df[col].dtype}")
        print(f"  Sample value: {df[col].iloc[0]}")

Before conversion:
  Order Date: object
  Sample value: 11/8/2017
  Ship Date: object
  Sample value: 11/11/2017


In [19]:
# TODO: Convert date columns to datetime
# Look at the column names printed above and convert them.
# The column names may vary depending on the dataset version.
#
# Example pattern:
# df['Column Name'] = pd.to_datetime(df['Column Name'])

# Your code here:
# Convert date columns to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

print("\nAfter conversion:")
for col in df.columns:
    if 'date' in col.lower():
        print(f"  {col}: {df[col].dtype}")
        print(f"  Sample value: {df[col].iloc[0]}")



After conversion:
  Order Date: datetime64[ns]
  Sample value: 2017-11-08 00:00:00
  Ship Date: datetime64[ns]
  Sample value: 2017-11-11 00:00:00


In [20]:
# TODO: Verify the conversion worked
# Print dtypes for the date columns to confirm they are datetime64

# Your code here:

print("After conversion:")
for col in df.columns:
    if 'date' in col.lower():
        print(f"  {col}: {df[col].dtype}")


After conversion:
  Order Date: datetime64[ns]
  Ship Date: datetime64[ns]


---
## Task 5: Derived Features (TODO)

Create customer-level summary features. These are the building blocks for customer segmentation (Activity 4).

You need to create:
- **total_spending**: Total sales per customer
- **order_frequency**: Number of orders per customer
- **avg_order_value**: Average sales amount per order per customer

Hint: Use `df.groupby('Customer ID')` (or whatever the customer ID column is named).

In [10]:
# First, identify the right column names
print("All columns:")
for col in df.columns:
    print(f"  {col}")

All columns:
  Row ID
  Order ID
  Order Date
  Ship Date
  Ship Mode
  Customer ID
  Customer Name
  Segment
  Country
  City
  State
  Postal Code
  Region
  Product ID
  Category
  Sub-Category
  Product Name
  Sales
  Quantity
  Discount
  Profit


In [21]:
# TODO: Create total_spending per customer
# Hint: df.groupby('Customer ID')['Sales'].sum()
#
# Replace column names below with the actual names from your dataset.

# customer_spending = df.groupby('???')['???'].sum()
# customer_spending.name = 'total_spending'

# Your code here:
# Create total_spending per customer
customer_spending = df.groupby('Customer ID')['Sales'].sum()
customer_spending.name = 'total_spending'

# Preview
customer_spending.head()

,total_spending
Customer ID,
AA-10315,5563.560
AA-10375,1056.390
AA-10480,1790.512
AA-10645,5086.935
AB-10015,886.156


In [22]:
# TODO: Create order_frequency per customer
# Hint: Count the number of rows (orders) per customer.
# Use .groupby(...).size() or .groupby(...)['some_col'].count()

# Your code here:
# Create order_frequency per customer (number of unique orders)
order_frequency = df.groupby('Customer ID')['Order ID'].nunique()
order_frequency.name = 'order_frequency'

# Preview
order_frequency.head()

,order_frequency
Customer ID,
AA-10315,5
AA-10375,9
AA-10480,4
AA-10645,6
AB-10015,3


In [24]:
order_frequency = order_frequency.reset_index()

In [25]:
# TODO: Create avg_order_value per customer
# Hint: Use .groupby(...)['Sales'].mean()

# Your code here:
# Step 1: total sales per order
order_totals = df.groupby(['Customer ID', 'Order ID'])['Sales'].sum()

# Step 2: average order value per customer
avg_order_value = order_totals.groupby('Customer ID').mean()
avg_order_value.name = 'avg_order_value'

# Preview
avg_order_value.head()


,avg_order_value
Customer ID,
AA-10315,1112.712000
AA-10375,117.376667
AA-10480,447.628000
AA-10645,847.822500
AB-10015,295.385333


In [26]:
# TODO: Combine all three into a single customer-level DataFrame
# Hint: Use pd.concat([series1, series2, series3], axis=1)
# or create them all at once with .groupby(...).agg(...)

# customer_summary = pd.concat([customer_spending, order_freq, avg_order], axis=1)
# customer_summary.columns = ['total_spending', 'order_frequency', 'avg_order_value']

# Your code here:
# Order-level totals first
order_totals = df.groupby(['Customer ID', 'Order ID'])['Sales'].sum().reset_index()

# Then customer-level metrics
customer_summary = order_totals.groupby('Customer ID').agg(
    total_spending=('Sales', 'sum'),
    order_frequency=('Order ID', 'nunique'),
    avg_order_value=('Sales', 'mean')
).reset_index()

customer_summary.head()


,Customer ID,total_spending,order_frequency,avg_order_value
0,AA-10315,5563.560,5,1112.712000
1,AA-10375,1056.390,9,117.376667
2,AA-10480,1790.512,4,447.628000
3,AA-10645,5086.935,6,847.822500
4,AB-10015,886.156,3,295.385333


In [27]:
# TODO: Display the first 10 rows of your customer summary and .describe()

# Your code here:
# Display first 10 rows
print(customer_summary.head(10))

# Summary statistics
print("\nSummary stats:")
print(customer_summary.describe())

  Customer ID  total_spending  order_frequency  avg_order_value
0    AA-10315        5563.560                5      1112.712000
1    AA-10375        1056.390                9       117.376667
2    AA-10480        1790.512                4       447.628000
3    AA-10645        5086.935                6       847.822500
4    AB-10015         886.156                3       295.385333
5    AB-10060        7755.620                8       969.452500
6    AB-10105       14473.571               10      1447.357100
7    AB-10150         966.710                5       193.342000
8    AB-10165        1113.838                8       139.229750
9    AB-10255         914.532                9       101.614667

Summary stats:
       total_spending  order_frequency  avg_order_value
count      793.000000       793.000000       793.000000
mean      2917.600051         6.696091       459.532952
std       2717.903575        11.149793       433.604846
min          4.833000         1.000000         2.416500


---
## Task 6: Save Cleaned Data (TODO)

Save the cleaned DataFrame to a new CSV file. Never overwrite the original.

In [29]:
# TODO: Save the cleaned main DataFrame
# df.to_csv('superstore_cleaned.csv', index=False)

# TODO: Save the customer summary DataFrame
# customer_summary.to_csv('customer_summary.csv')

# Your code here:
# Save the cleaned main DataFrame
df.to_csv('superstore_cleaned.csv', index=False)

# Save the customer summary DataFrame
customer_summary.to_csv('customer_summary.csv', index=False)

---
## Reflection

Answer these in a text cell or comments:

1. Why did we use median instead of mean for filling numerical missing values?
2. What is the difference between the row-level DataFrame (one row per order) and the customer-level summary? When would you use each?
3. If two rows have identical values in every column, are they always true duplicates? When might they not be?